# Lab 5: Query Decomposition

**Workshop 2, block 4. 20 minutes.**

Some answers are not written down anywhere and have to be assembled. Two
causes:

- **Compound questions.** One embedding of a two-part question lands
  between both topics and close to neither.
- **Counting questions.** "How many workshop posters for 2025" has the
  answer 31, and no chunk contains 31. Worse, with k=5 your bot has seen
  at most five posters and cannot report 31 whatever the embeddings do.

1. Record your current answer and its timing
2. Add a call that splits the question
3. Retrieve for each part and label the results
4. Answer from the combined context, and time it again
5. Run it on a Lv1 question too

> **Before you start:** `data/chroma` with the image descriptions added in lab 4, and `dev_set.json`.
>
> **When you finish:** A working decomposition function, and a measurement of what it costs you in seconds.

---
## Setup

Run these two cells first. They are identical in every lab, so each
notebook works on its own.

Your key is entered with `getpass`: not echoed, not written to disk, and
gone when the kernel stops. **Do not commit a notebook with a key
visible in its output.**

In [ ]:
# pip install openai chromadb beautifulsoup4 requests python-dotenv

import getpass, json, os, re, time, statistics
from pathlib import Path
from openai import AzureOpenAI

# Reads .env if you have one, otherwise uses the defaults below, so the
# notebook runs either way. Copy .env.example to .env to override.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview")
CHAT_BASE   = os.getenv("AZURE_CHAT_BASE",  "https://api-iw.azure-api.net/sig-shared-jpeast-increased")
EMBED_BASE  = os.getenv("AZURE_EMBED_BASE", "https://api-iw.azure-api.net/sig-embedding")

CHAT_DEPLOYMENT   = os.getenv("CHAT_DEPLOYMENT",   "gpt-4o-mini")
VISION_DEPLOYMENT = os.getenv("VISION_DEPLOYMENT", "gpt-5-mini")
EMBED_DEPLOYMENT  = os.getenv("EMBED_DEPLOYMENT",  "text-embedding-3-small")

# This gateway takes the FULL path as the endpoint: deployment, operation
# and api-version included. So a client is bound to one deployment, and
# we need three of them.
#
# The chat route has no /openai segment, the embedding route does. That
# asymmetry is real, so do not tidy them into matching.

CHAT_URL   = f"{CHAT_BASE}/deployments/{CHAT_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
VISION_URL = f"{CHAT_BASE}/deployments/{VISION_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
EMBED_URL  = f"{EMBED_BASE}/openai/deployments/{EMBED_DEPLOYMENT}/embeddings?api-version={API_VERSION}"

# .strip() matters: pasting into a prompt often picks up a trailing
# newline, and that alone produces a 401.
KEY = (os.getenv("AZURE_OPENAI_KEY")
       or getpass.getpass("Azure OpenAI key: ")).strip()


def _client(url):
    return AzureOpenAI(azure_endpoint=url, api_key=KEY, api_version=API_VERSION)


chat_client   = _client(CHAT_URL)
vision_client = _client(VISION_URL)
embed_client  = _client(EMBED_URL)

# Each is tested separately so one failure does not hide the others.
for label, fn in [
    ("chat  ", lambda: chat_client.chat.completions.create(
        model=CHAT_DEPLOYMENT, max_tokens=5,
        messages=[{"role": "user", "content": "Reply with one word: connected"}]
     ).choices[0].message.content.strip()),
    ("vision", lambda: vision_client.chat.completions.create(
        model=VISION_DEPLOYMENT, max_tokens=5,
        messages=[{"role": "user", "content": "Reply with one word: connected"}]
     ).choices[0].message.content.strip()),
    ("embed ", lambda: f"{len(embed_client.embeddings.create(model=EMBED_DEPLOYMENT, input=['test']).data[0].embedding)} dimensions"),
]:
    try:
        print(f"{label}  OK      {fn()}")
    except Exception as e:
        print(f"{label}  FAILED  {type(e).__name__}: {str(e)[:110]}")

# 401 means the path is right and the key is wrong.
# 404 means the path is wrong, not the key.

In [ ]:
# ---- helpers ------------------------------------------------------

def chat(messages, model=None, temperature=0.0, max_tokens=512):
    r = chat_client.chat.completions.create(
        model=model or CHAT_DEPLOYMENT, messages=messages,
        temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content or ""


def ask(prompt, system=None, **kw):
    msgs = ([{"role": "system", "content": system}] if system else [])
    return chat(msgs + [{"role": "user", "content": prompt}], **kw)


def embed(texts, batch_size=256):
    """Embed a LIST of strings. One call per string is the slow mistake:
    3,000 round trips at ~200ms each is ten minutes of network wait."""
    texts = [t.replace("\n", " ") for t in texts]
    out = []
    for i in range(0, len(texts), batch_size):
        r = embed_client.embeddings.create(model=EMBED_DEPLOYMENT, input=texts[i:i + batch_size])
        out.extend(d.embedding for d in r.data)
    return out


def chunk(text, size=800, overlap=100):
    """Fixed-size chunks with overlap. Defaults to start from, not
    recommended values."""
    text = " ".join(text.split())
    step = size - overlap
    return [text[i:i + size] for i in range(0, len(text), step)
            if text[i:i + size].strip()]


# One PersistentClient per path, cached for the life of this kernel.
# chromadb caches internal state per path, so deleting the folder and
# opening a fresh PersistentClient while an earlier one from this same
# session is still alive corrupts the connection: you get "attempt to
# write a readonly database" or "database is locked" on the very next
# call. Rebuilding your index more than once per session, which the
# "change one setting, re-run" loop asks you to do, hits this every
# time with the naive version.
_stores = {}


def get_store(path="data/chroma", name="workshop", reset=False):
    import chromadb, shutil
    from pathlib import Path as _P

    if path not in _stores:
        # First time this path is opened in this session. Safe to wipe a
        # stale, wrong-chromadb-version index here, since no client for
        # this path exists in this process yet.
        if reset and _P(path).exists():
            shutil.rmtree(path)
        try:
            _stores[path] = chromadb.PersistentClient(path=path)
        except KeyError as e:
            raise RuntimeError(
                f"chromadb cannot read the index at {path} ({e}). It was built by "
                f"a different chromadb version. Delete that folder and rebuild, or "
                f"install the pinned version from requirements.txt."
            ) from None

    client = _stores[path]
    if reset:
        # Reset now means delete-and-recreate the COLLECTION on the same
        # client, not delete-and-recreate the DIRECTORY under it. This is
        # what actually avoids the readonly/locked error on every rebuild
        # after the first.
        try:
            client.delete_collection(name)
        except Exception:
            pass
    return client.get_or_create_collection(name)


def add_to_store(store, texts, metadatas, ids=None, batch_size=128):
    ids = ids or [f"c{i}" for i in range(len(texts))]
    for i in range(0, len(texts), batch_size):
        sl = slice(i, i + batch_size)
        store.add(ids=ids[sl], documents=texts[sl],
                  embeddings=embed(texts[sl]), metadatas=metadatas[sl])


def query(store, question, k=5, where=None):
    """The k nearest chunks. Chroma returns squared L2, so lower is closer."""
    r = store.query(query_embeddings=embed([question]), n_results=k,
                    where=where or None)
    return [{"text": d, "metadata": m, "distance": dist}
            for d, m, dist in zip(r["documents"][0], r["metadatas"][0],
                                  r["distances"][0])]


def show(chunks, chars=200):
    if not chunks:
        print("  (nothing returned)")
        return
    for c in chunks:
        print(f"  {c['distance']:.3f}  {c['metadata'].get('url', '?')}")
        print(f"         {c['text'][:chars].strip()}\n")


def timed(fn, *a, **kw):
    t0 = time.time()
    return fn(*a, **kw), time.time() - t0


def normalise(s):
    return re.sub(r"[^a-z0-9 ]", " ", (s or "").lower())


def answer_present(expected, chunks):
    """Was the expected answer anywhere in the retrieved text? Crude, and
    enough to tell a retrieval failure from a prompt failure."""
    hay  = normalise(" ".join(c["text"] for c in chunks))
    need = normalise(expected).strip()
    if need and need in hay:
        return True
    terms = [t for t in need.split() if len(t) > 3]
    return bool(terms) and sum(t in hay for t in terms) / len(terms) >= 0.8


def load_dev_set(path="dev_set.json"):
    return json.loads(Path(path).read_text())


def describe_image(image_path, prompt, model=None):
    """One image plus an instruction. There is deliberately no default
    prompt: writing it is the lab 4 exercise."""
    import base64, mimetypes
    p    = Path(image_path)
    mime = mimetypes.guess_type(p.name)[0] or "image/jpeg"
    b64  = base64.b64encode(p.read_bytes()).decode()
    r = vision_client.chat.completions.create(
        model=model or VISION_DEPLOYMENT, max_tokens=800,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
        ]}])
    return r.choices[0].message.content or ""

print("helpers loaded")

In [ ]:
store = get_store("data/chroma", name="workshop")
dev   = load_dev_set("dev_set.json")
print(f"{store.count()} chunks indexed")

lv4 = [d for d in dev if d["level"] == 4]
for d in lv4:
    print(f"  {d['question']}\n     -> {d['answer']}\n")

---
## Step 1: Baseline

Pick a Lv4 question your bot gets wrong. Record the answer **and the
seconds**, because you are comparing both.

In [ ]:
SYSTEM = """Answer only from the context. Reply with the answer only.
If the question asks how many, reply with a number."""

def answer_simple(question, k=5):
    chunks  = query(store, question, k=k)
    context = "\n\n".join(c["text"] for c in chunks)
    return chat([
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]).strip()


Q = lv4[0]["question"]
reply, secs = timed(answer_simple, Q)
print(f"Q: {Q}\nA: {reply}\n   {secs:.1f}s")

---
## Steps 2 and 3: Split, retrieve per part, label

Labelling which sub-question each chunk came from matters. Without
labels the model sees one undifferentiated block of text and often
answers only the first part.

In [ ]:
def split_question(question):
    """One cheap call. Returns a list of sub-questions."""
    out = ask(f"Split this question into the separate factual questions needed "
              f"to answer it. One per line, no numbering, no commentary. "
              f"If it is already a single question, return it unchanged."
              f"\n\n{question}", max_tokens=200)
    return [l.strip(" -0123456789.") for l in out.splitlines() if l.strip()]


def answer_decomposed(question, k=5, verbose=True):
    parts = split_question(question)
    if verbose:
        print("split into:")
        for p in parts:
            print("   -", p)

    blocks = []
    for i, part in enumerate(parts, 1):
        joined = "\n".join(c["text"] for c in query(store, part, k=k))
        blocks.append(f"[from sub-question {i}: {part}]\n{joined}")

    return chat([
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": "Context:\n" + "\n\n".join(blocks)
                                    + f"\n\nQuestion: {question}"},
    ]).strip()


reply2, secs2 = timed(answer_decomposed, Q)
print(f"\nA: {reply2}\n   {secs2:.1f}s   (was {secs:.1f}s)")

---
## Step 4: Counting needs filtering, not searching

Decomposition does not fix counting. To count posters from 2025 you have
to select **all** of them, then count the set. That is a database
operation, not a search, and it depends entirely on the metadata you
stored when indexing.

Lab 4 tagged your image chunks `kind="image"`. If you never recorded
page type or year on the text chunks, there is nothing to filter on and
adding the field now means rebuilding the index.

In [ ]:
def answer_by_counting(question, where, k=200):
    chunks = query(store, question, k=k, where=where)
    print(f"{len(chunks)} items matched {where}")
    listing = "\n".join(f"- {c['text'][:120]}" for c in chunks)
    return chat([
        {"role": "system", "content":
         "Count the items listed. List what you counted, then give the number "
         "on its own final line."},
        {"role": "user", "content": f"Items:\n{listing}\n\nQuestion: {question}"},
    ], max_tokens=700).strip()


try:
    print(answer_by_counting("How many images have you indexed?",
                             where={"kind": "image"}))
except Exception as exc:
    print("Filter failed:", exc)
    print("Check which metadata fields you actually stored.")

---
## Step 5: Now run it on a Lv1 question

This is the point of the lab. Decomposition typically triples the time.
On a question that did not need it, the score is unchanged and you spent
the seconds anyway.

Thirty seconds is a hard limit, and a timed-out question scores zero
however good the answer would have been.

In [ ]:
lv1 = [d for d in dev if d["level"] == 1][0]["question"]

a1, t1 = timed(answer_simple, lv1)
a2, t2 = timed(answer_decomposed, lv1, verbose=False)

print(f"Q: {lv1}\n")
print(f"  simple       {t1:.1f}s   {a1}")
print(f"  decomposed   {t2:.1f}s   {a2}")
print(f"\n  {t2/max(t1,0.01):.1f}x the time for the same answer.")

---
## What follows: routing

Classify the question with one cheap call, then dispatch. Easy questions
take the fast path, hard ones pay for the slow one.

In [ ]:
def route(question):
    lv = ask("Classify this question. Reply with one number only.\n"
             "1 = general knowledge, no lookup needed\n"
             "2 = a single fact from one page\n"
             "3 = a fact that lives in an image\n"
             "4 = needs several facts combined, or a count\n\n"
             f"{question}", max_tokens=4).strip()
    return lv[:1] if lv[:1] in "1234" else "2"


for d in dev[:6]:
    print(f"  predicted Lv{route(d['question'])}   actual Lv{d['level']}   "
          f"{d['question'][:52]}")

---
## Going further

- **Retrieve wide, then filter.** Fetch 30 to 50 chunks and ask the
  model to discard the irrelevant ones. Models judge relevance better
  than distance ranks it.
- **Evidence first.** Have the model list what it found before stating
  an answer. Counting accuracy improves and failures become visible.
- **Run sub-queries in parallel** with `asyncio` or a thread pool. A
  five-step sequential chain becomes roughly the cost of its slowest
  step.